# DEFHS — New independent Hindi-adapted emotion experts (October 12, 2026)

**Scientific role:** A genuinely separate speech-emotion decision signal. No retraining or reweighting of System E.

**Hypotheses:** Four pretrained cross-lingual audio-emotion checkpoints may provide genuinely complementary acoustic and prosodic evidence. QnAce reports Hindi support; its loader compatibility must be established experimentally. The others are cross-lingual or IEMOCAP task-specific baselines. These are NOT validated challengers yet.

**Rules:** Internet ON; GPU ON; Kaggle competition dataset added as input. The notebook produces expert evidence and NEVER submits or overrides the 78 certified retrieval labels. Evaluate external models on all official 826 labeled clips and then all 209 test clips; no hidden labels used. Output is a CSV with confidence/margins and a descriptive report. Do not claim PRIMARY_CV or private score for these zero-shot predictions.

**Stop rule:** No model will be proposed for E replacement without credible training-label evidence and paired validation against original System E probabilities. The model output is independent evidence only.

In [ ]:
import os, sys, subprocess, importlib.util
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers>=4.47,<5","soundfile","safetensors"])
import gc, json, warnings, time, glob, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch, soundfile as sf
from sklearn.metrics import f1_score, confusion_matrix
from transformers import AutoFeatureExtractor, AutoProcessor, AutoModelForAudioClassification
print('Python',sys.version,'Torch',torch.__version__,'GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Transformers',__import__('transformers').__version__)
assert torch.cuda.is_available(), 'This mission requires Kaggle GPU; enable accelerator in notebook Settings'
OUT=Path('/kaggle/working/defhs_independent_experts');OUT.mkdir(exist_ok=True,parents=True)
LABS=['angry','happy','neutral','sad']


In [ ]:
# Locate the actual official competition input, never fetch substitutes.
roots=[p for p in Path('/kaggle/input').rglob('train.csv') if 'SRCASW' in str(p) or 'dataverse' in str(p).lower() or (p.parent/'test.csv').exists()]
roots=list(dict.fromkeys(str(p.parent) for p in roots))
print('candidate input roots:',roots)
assert roots, 'Add the official competition data as Kaggle notebook input'
root=Path(roots[0]); tr=pd.read_csv(root/'train.csv'); te=pd.read_csv(root/'test.csv')
print('train',tr.shape,tr.columns.tolist(),'test',te.shape,te.columns.tolist(),'data_root',root)
assert len(tr)==826 and len(te)==209, 'Unexpected data sizes'
# The canonical DEFHS contract uses numerical Id, emotion and audio_000xxx.wav.
def idcol(df):
    cols={x.lower():x for x in df.columns}
    return cols.get('id',cols.get('audio_id'))
def labelcol(df):
    cols={x.lower():x for x in df.columns}
    return cols.get('emotion',cols.get('label'))
tr_id=idcol(tr);te_id=idcol(te);yl=labelcol(tr)
assert tr_id and te_id and yl, (tr.columns.tolist(),te.columns.tolist())
train_ids=tr[tr_id].astype(int).to_numpy();test_ids=te[te_id].astype(int).to_numpy()
assert len(set(train_ids))==826 and len(set(test_ids))==209 and set(train_ids).isdisjoint(set(test_ids))
y=tr[yl].astype(str).str.lower().str.strip().to_numpy()
assert set(y)==set(LABS),set(y)
all_ids=np.concatenate([train_ids,test_ids]); split=np.array(['train']*826+['test']*209)
paths=[]
for i,t in zip(all_ids,split):
    fn=f'audio_{int(i):06}.wav'
    choices=list(root.rglob(fn))
    assert len(choices)==1,(fn,choices)
    paths.append(choices[0])
print('OFFICIAL AUDIO VERIFIED',len(paths), 'FIRST',paths[0],'LAST',paths[-1])


In [ ]:
# Read short Hindi speech clips. No trimming/augmenting that could change emotion.
from scipy.signal import resample_poly
WAV=[]
for p in paths:
    x,sr=sf.read(p,dtype='float32',always_2d=False)
    if x.ndim>1: x=x.mean(axis=1)
    if sr!=16000:
        import math
        g=math.gcd(sr,16000);x=resample_poly(x,16000//g,sr//g).astype(np.float32)
    assert len(x)>160 and np.all(np.isfinite(x)),p
    WAV.append(x)
print('AUDIO READ',len(WAV),'max seconds',max(len(x) for x in WAV)/16000,'median',np.median([len(x) for x in WAV])/16000)


In [ ]:
# Independent pretrained models: do NOT re-run emotion2vec+ or AST; already rejected.
# FairHindiSER publishes separate PyTorch .pt state_dicts but no importable custom FairSERModel class, so avoid invalid loading.
MODELS=[
  'abdullahjamil42/QnAce-Voice-Model',           # Hindi / Urdu / English 6 emotion labels; assess loader support
  'superb/wav2vec2-base-superb-er',              # independently trained IEMOCAP 4-class ER, verified HF architecture
  'Aniemore/wav2vec2-emotion-v1-crosslingual',   # cross-language 7 labels, standard HF loader
  'Aniemore/wav2vec2-bert-base-s-emotion-v1-crosslingual', # independent architecture; stop if unneeded
]
ALIAS={
    'angry':'angry','anger':'angry','mad':'angry','ang':'angry',
    'happy':'happy','hap':'happy','happiness':'happy','joy':'happy','joyful':'happy','excited':'happy','enthusiasm':'happy',
    'neutral':'neutral','neu':'neutral','calm':'neutral',
    'sad':'sad','sadness':'sad'
}
def map_target(s):
    k=''.join(x for x in str(s).lower() if x.isalpha())
    return ALIAS.get(k)
allresults=[];model_status=[]
for model_name in MODELS:
    print('\nBEGIN',model_name,flush=True)
    t0=time.time()
    try:
        try: processor=AutoProcessor.from_pretrained(model_name,trust_remote_code=True)
        except Exception: processor=AutoFeatureExtractor.from_pretrained(model_name,trust_remote_code=True)
        model=AutoModelForAudioClassification.from_pretrained(model_name,trust_remote_code=True).eval().to('cuda')
        id2label={int(k):str(v) for k,v in model.config.id2label.items()}
        names={int(k):map_target(v) for k,v in id2label.items()}
        used=[k for k,v in names.items() if v in LABS]
        assert set(names[k] for k in used)==set(LABS), ('Cannot safely map all 4 emotions',id2label)
        order=[[k for k,v in names.items() if v==lab] for lab in LABS]
        print('Labels',id2label,'mapped indices',order)
        probs=[];masses=[]
        bs=4
        for start in range(0,len(WAV),bs):
            seq=WAV[start:start+bs]
            inp=processor(seq,sampling_rate=16000,return_tensors='pt',padding=True)
            inp={k:v.to('cuda') for k,v in inp.items() if torch.is_tensor(v) and k in ('input_values','input_features','attention_mask')}
            with torch.inference_mode():
                logits=model(**inp).logits.float()
                raw=logits.softmax(dim=-1).cpu().numpy()
            assert raw.shape[1]==len(id2label),(raw.shape,id2label)
            sub=np.stack([raw[:,inds].sum(axis=1) for inds in order],axis=1)
            mass=sub.sum(axis=1)
            p=sub/np.maximum(mass[:,None],1e-8)
            probs.append(p);masses.append(mass)
            if start and start%256<bs: print(model_name,'processed',start+len(seq),'/',len(WAV),flush=True)
        P=np.concatenate(probs);mass=np.concatenate(masses)
        pred=np.asarray(LABS)[P.argmax(axis=1)]
        f1=f1_score(y,pred[:826],labels=LABS,average='macro')
        cm=confusion_matrix(y,pred[:826],labels=LABS)
        result={'model':model_name,'status':'PASS','train_zero_shot_macro_f1':float(f1),
                'per_class_f1':{c:float(f1_score(y==c,pred[:826]==c)) for c in LABS},
                'confusion':cm.tolist(),'mean_4class_original_mass':float(mass.mean()),
                'time_seconds':round(time.time()-t0,1),'n_train':826,'n_test':209}
        df=pd.DataFrame({'Id':all_ids,'split':split,'model':model_name,'prediction':pred,'target_mass':mass})
        for j,c in enumerate(LABS):df['p_'+c]=P[:,j]
        df.to_csv(OUT/(model_name.split('/')[-1]+'_all1035.csv'),index=False,float_format='%.12g')
        allresults.append(df);model_status.append(result)
        print('RESULT',json.dumps(result,indent=2),flush=True)
        del processor,model,inp,logits;gc.collect();torch.cuda.empty_cache()
    except Exception as exc:
        fail={'model':model_name,'status':'MODEL_LOADING_OR_INFERENCE_FAILED','reason':repr(exc),'time_seconds':round(time.time()-t0,1)}
        model_status.append(fail);print('FAILED',fail,flush=True)
        try: del model
        except: pass
        gc.collect();torch.cuda.empty_cache()
    (OUT/'STATUS.json').write_text(json.dumps(model_status,indent=2)+'\n')
assert allresults, 'No pretrained independent model loaded; check failure reasons in STATUS.json'
print('SUCCESSFUL_EXPERTS',len(allresults))


In [ ]:
# Concise side-by-side disagreement evidence; no system override.
wide=pd.DataFrame({'Id':all_ids,'split':split})
for j,df in enumerate(allresults):
    a='expert'+str(j)
    wide[a+'_name']=df.model.iloc[0]
    wide[a+'_prediction']=df.prediction.to_numpy()
    wide[a+'_max_p']=df[['p_'+x for x in LABS]].to_numpy().max(axis=1)
    wide[a+'_target_mass']=df.target_mass.to_numpy()
wide.to_csv(OUT/'EXPERTS_ALL1035.csv',index=False)
(OUT/'INPUT_MANIFEST.json').write_text(json.dumps({'competition_root':str(root),'models':MODELS,'train_n':826,'test_n':209,'execution':model_status,'no_submission':True,'no_test_labels':True,'no_retrieval_override':True},indent=2)+'\n')
print('RESULTS READY IN',OUT)
print(wide[wide.split=='test'].head(20).to_string(index=False))


In [ ]:
# Independent paired corrections vs frozen acoustic reference (0.7768156647), and unlabeled E target disagreements.
# Original coefficients/labels/folds are embedded once with exact provenance; NO leaderboard or test labels.
import base64,zlib,io
REFPAYLOAD='...'
q=np.load(io.BytesIO(zlib.decompress(base64.b64decode(REFPAYLOAD))),allow_pickle=False)
from sklearn.metrics import f1_score
frozen_ids=q['Id'];folds=q['fold'];pairp=q['pair_p'];lab=np.asarray(LABS)
fm=pd.DataFrame({'Id':frozen_ids,'fold':folds,'pair_prediction':lab[pairp.argmax(1)]})
assert len(fm)==826 and abs(f1_score(tr.set_index(tr_id).loc[frozen_ids,yl].astype(str).str.lower(),lab[pairp.argmax(1)],labels=LABS,average='macro')-.7768156647359765)<1e-12
base_test=pd.DataFrame({'Id':q['E_Id'],'E_prediction':q['E_label'].astype(str)})
retrieval=set(map(int,q['certified']))
reports=[]
for r in allresults:
    native=r[r.split=='train'].set_index('Id').loc[frozen_ids]
    ep=native.prediction.to_numpy()
    baseline=fm['pair_prediction'].to_numpy()
    real=tr.set_index(tr_id).loc[frozen_ids,yl].astype(str).str.lower().to_numpy()
    a=(baseline==real);b=(ep==real)
    folds_detail=[{'fold':f,'expert_f1':float(f1_score(real[folds==f],ep[folds==f],labels=LABS,average='macro')),'pair_f1':float(f1_score(real[folds==f],baseline[folds==f],labels=LABS,average='macro'))} for f in range(5)]
    tar=r[r.split=='test'][['Id','prediction','target_mass']].merge(base_test,on='Id',validate='one_to_one')
    tar['certified']=tar.Id.isin(retrieval)
    tar['diff_E']=tar.prediction!=tar.E_prediction
    disagreements=tar[tar.diff_E]
    res={'model':str(r.model.iloc[0]),'external_zero_shot_train_f1':float(f1_score(real,ep,labels=LABS,average='macro')),
         'pair_reference_f1':.7768156647359765,
         'pair_wrong_new_correct':int((~a & b).sum()),'pair_correct_new_wrong':int((a & ~b).sum()),
         'both_wrong':int((~a & ~b).sum()),'both_correct':int((a & b).sum()),
         'folds':folds_detail,
         'test_disagreements_E':int(tar.diff_E.sum()),'noncertified_disagreements_E':int((tar.diff_E&~tar.certified).sum()),
         'certified_disagreements_prohibited_as_override':int((tar.diff_E&tar.certified).sum())}
    reports.append(res)
    disagreements.to_csv(OUT/('DISAGREEMENTS_'+r.model.iloc[0].split('/')[-1]+'.csv'),index=False)
    print(json.dumps(res,indent=2))
(OUT/'PAIRED_DIAGNOSTICS.json').write_text(json.dumps(reports,indent=2)+'\n')
print('No automatic submission, no promotion, no test-label assertion')


In [ ]:
# One predeclared, low-capacity conditional-information test across the new frozen experts.
# No hidden labels, no label leakage across five frozen original outer folds; baseline is exact historical pair.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
if len(allresults)>=2:
    ordered=allresults
    train_features=np.column_stack([
        d[d.split=='train'].set_index('Id').loc[frozen_ids,['p_'+c for c in LABS]].to_numpy(float)
        for d in ordered])
    test_features=np.column_stack([
        d[d.split=='test'].set_index('Id').loc[q['E_Id'],['p_'+c for c in LABS]].to_numpy(float)
        for d in ordered])
    real=tr.set_index(tr_id).loc[frozen_ids,yl].astype(str).str.lower().to_numpy()
    oof=np.zeros((826,4))
    for f in range(5):
        is_tr=(folds!=f)
        m=LogisticRegression(C=0.5,max_iter=2000,random_state=20261012)
        m.fit(train_features[is_tr],real[is_tr]);oof[~is_tr]=m.predict_proba(train_features[~is_tr])
    pred=np.asarray(LABS)[oof.argmax(1)]
    paired=np.asarray(LABS)[pairp.argmax(1)]
    fold_deltas=[]
    for f in range(5):
        mask=(folds==f)
        fold_deltas.append(float(f1_score(real[mask],pred[mask],labels=LABS,average='macro')-f1_score(real[mask],paired[mask],labels=LABS,average='macro')))
    score=float(f1_score(real,pred,labels=LABS,average='macro'))
    corrected=int(np.sum((paired!=real)&(pred==real)))
    broken=int(np.sum((paired==real)&(pred!=real)))
    report={'status':'CROSS_FITTED_COMPARISON_ONLY','model_names':[str(d.model.iloc[0]) for d in ordered],
            'OOF_macro_f1':score,'pair_baseline':.7768156647359765,
            'System_E_historical_macro_f1':.8774091527555827,
            'fold_deltas_vs_pair':fold_deltas,'corrections_vs_pair':corrected,'breakages_vs_pair':broken,
            'no_hidden_labels':True,'no_automatic_submission':True}
    (OUT/'FUSION_DIAGNOSTICS.json').write_text(json.dumps(report,indent=2)+'\n')
    print('NEW EXPERT FUSION',json.dumps(report,indent=2))
    # Only produce test predictions for independent review. 78 certified identities locked.
    final=LogisticRegression(C=0.5,max_iter=2000,random_state=20261012).fit(train_features,real)
    ptest=final.predict_proba(test_features)
    labels=np.asarray(LABS)[ptest.argmax(1)]
    df=pd.DataFrame({'Id':q['E_Id'],'candidate':labels,'original_E':q['E_label'].astype(str),
                     'certified_retrieval':pd.Series(q['E_Id']).isin(retrieval).to_numpy()})
    df['safe_diagnostic_label']=np.where(df.certified_retrieval,df.original_E,df.candidate)
    df['would_change_system_E']=df.safe_diagnostic_label!=df.original_E
    df.to_csv(OUT/'FUSION_CANDIDATE_DIAGNOSTIC_ONLY.csv',index=False)
    print('FUSION VS E: hypothetical differences',int(df.would_change_system_E.sum()),
          'on certified',int((df.would_change_system_E&df.certified_retrieval).sum()))
else:
    print('FUSION NOT ATTEMPTED: fewer than two compatible independent models loaded')
